# Comparing interatomic potentials for dislocations in fcc Ni

**Benchmark chain**

| Step | Quantity | Tool |
|---|---|---|
| 1 | lattice parameter $a_0$ | LAMMPS (`a0.in`) |
| 2 | Burgers vectors $b$, $b_p$ | Python |
| 3 | elastic constants $C_{11}, C_{12}, C_{44}$ | LAMMPS (`elastic.in`) |
| 4 | shear modulus $\mu$, Poisson ratio $\nu$ | atomman `ElasticConstants` |
| 5 | generalized stacking-fault (γ) surface, $\gamma_{usf}$, $\gamma_{isf}$ | atomman `StackingFault` + LAMMPS (`gsf.in`) |
| 6 | equilibrium partial separation $d_{sf}$ | Python ($\mu b^2/12\pi\gamma_{sf}$) + atomman `Stroh` (anisotropic) |
| 7 | periodic array of dislocations (PAD), partial separation | atomman `Dislocation` + LAMMPS (`pad.in`) + OVITO (CNA, DXA) |
| 8 | disregistry and CRSS | atomman `disregistry`, `SDVPN` |

Structures are ASE `Atoms` objects throughout: built with ASE (or atomman for defects), written to LAMMPS data files with ASE,
and analysed and drawn with OVITO.

Run the notebook once per potential (selected in **Setup**). Each run writes `results_<potential>.json` for the final comparison.

> **Structure only.** The LAMMPS scripts in `scripts/` will be provided separately. Each section states its **script contract**: what the notebook passes in as `-var` variables, and what it reads back.

## 0. Setup
### 0.1 Install (Google Colab)
The Python packages all come from PyPI, so nothing needs compiling. The LAMMPS wheel is linked against MPICH, and the `[mpi]` extra installs that library. LAMMPS still runs as a **single serial process**: `mpirun` is never used.
OVITO renders images through Vulkan; Colab has no Vulkan driver, so the second line installs Mesa's software driver (no GPU needed).

In [ ]:
%pip install -q "lammps[mpi]==2025.7.22.4.0" "atomman==1.5.4" "ase==3.29.0" "ovito==3.16.1"
!apt-get -qq update > /dev/null && apt-get -qq install -y mesa-vulkan-drivers > /dev/null   # software Vulkan driver, needed by OVITO's renderer

### 0.2 Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# The LAMMPS wheel links against MPICH's libmpi.so.12, which pip installs outside the loader's search path
# (on Colab: /usr/local/lib, whose loader cache is not refreshed). Load it explicitly before importing lammps;
# also export its folder so that `!lmp ...` shell cells find it.
import os, sys, ctypes, importlib.metadata
_libmpi = next(f.locate().resolve() for f in importlib.metadata.files('mpich') if f.name == 'libmpi.so.12')
ctypes.CDLL(str(_libmpi), mode=ctypes.RTLD_GLOBAL)
os.environ['LD_LIBRARY_PATH'] = f"{_libmpi.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}"
# `!lmp` needs the folder of this kernel's Python on PATH: .venv/bin in a local venv
# (Jupyter does not activate it), /usr/local/bin on Colab.
os.environ['PATH'] = f"{Path(sys.executable).parent}:{os.environ.get('PATH', '')}"

import lammps
from lammps import lammps as LAMMPS

import atomman as am
import atomman.unitconvert as uc

import ase, ase.build, ase.io
import ovito, ovito.gui, ipywidgets
from ovito.io.ase import ase_to_ovito
from ovito.pipeline import Pipeline, StaticSource
from ovito.modifiers import (CommonNeighborAnalysisModifier as CNA, DislocationAnalysisModifier as DXA,
                             ExpressionSelectionModifier, DeleteSelectedModifier, ComputePropertyModifier)
from ovito.vis import Viewport, TachyonRenderer

print('LAMMPS', LAMMPS(cmdargs=['-screen', 'none', '-log', 'none']).version(), '| atomman', am.__version__,
      '| ASE', ase.__version__, '| OVITO', ovito.version_string)

### 0.3 Tutorial files
The potential files (`ni_potentials/`) and the LAMMPS scripts (`scripts/`) come from the course repository,
[github.com/TheNewPing/multi_scale_md_tutorial](https://github.com/TheNewPing/multi_scale_md_tutorial).
The cell below downloads the repository as a zip archive and extracts only these two folders into the current
directory, so it needs neither `git` nor a GitHub account.

* A folder that already exists is **not** downloaded again. This keeps any edits you made, and it means the cell
  does nothing when you run the notebook from a local clone of the repository.
* To discard your edits and get a fresh copy, run `fetch_course_files(force=True)`.

In [ ]:
import io, shutil, urllib.error, urllib.request, zipfile

REPO, BRANCH = 'TheNewPing/multi_scale_md_tutorial', 'master'
FOLDERS      = ['ni_potentials', 'scripts']        # extracted into the current directory

def fetch_course_files(folders=FOLDERS, repo=REPO, branch=BRANCH, force=False):
    todo = [f for f in folders if force or not Path(f).is_dir()]
    if not todo:
        print('already present:', ', '.join(f'{f}/' for f in folders))
        return
    url = f'https://github.com/{repo}/archive/refs/heads/{branch}.zip'
    try:
        with urllib.request.urlopen(url) as r:
            zf = zipfile.ZipFile(io.BytesIO(r.read()))
    except urllib.error.HTTPError as e:
        raise RuntimeError(f'download of {url} failed (HTTP {e.code}); is the repository public?') from None
    root = zf.namelist()[0].split('/')[0]           # GitHub puts everything under '<repo>-<branch>/'
    for folder in todo:
        members = [m for m in zf.namelist() if m.startswith(f'{root}/{folder}/') and not m.endswith('/')]
        if not members:
            print(f'WARNING: {folder}/ is not in {repo}@{branch}')
            continue
        shutil.rmtree(folder, ignore_errors=True)
        for m in members:
            dest = Path(m[len(root) + 1:])
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(zf.read(m))
        print(f'{folder}/:', ', '.join(sorted(str(Path(m).relative_to(f"{root}/{folder}")) for m in members)))

fetch_course_files()

SCRIPT_DIR = Path('scripts')            # a0.in, elastic.in, gsf.in, pad.in
WORK       = Path('work'); WORK.mkdir(exist_ok=True)

# Let LAMMPS also find the potential files that ship with the wheel by name (e.g. Ni_u3.eam)
os.environ['LAMMPS_POTENTIALS'] = str(Path(lammps.__file__).parent / 'share' / 'lammps' / 'potentials')

### 0.4 The interatomic potential: `potential.mod`
Every LAMMPS script reads the potential from one small file with `include ${potential_file}`, after the box has been created.
**To test a different potential, edit this file (or write another one) and re-run the notebook. Nothing else changes.**

A `.mod` file holds ordinary LAMMPS commands:

| line | meaning | what to change |
|---|---|---|
| `pair_style eam/alloy` | functional form of the potential | `eam/alloy` for setfl files (`*.eam.alloy`), `eam/fs` for Finnis–Sinclair setfl files, `eam` for single-element funcfl files (`*.eam`), `meam`, `snap`, … |
| `pair_coeff * * <file> Ni` | parameter file, then one element name per atom type | the file path; the element as it is named in that file. Every structure in this notebook has **one** atom type, so there is a single `Ni`. For `pair_style eam` the element name is omitted (`pair_coeff * * Ni_u3.eam`). |
| `mass * 58.6934` | atomic mass of every type (g/mol) | only for other elements; it plays no role in static relaxations |

Examples:
```
pair_style eam                                    # Foiles, Baskes & Daw (1986), file ships with LAMMPS
pair_coeff * * Ni_u3.eam

pair_style eam/alloy                              # Mishin et al. (2002)
pair_coeff * * ni_potentials/NiAl02.eam.alloy.in Ni

pair_style meam                                   # MEAM: library file + parameter file
pair_coeff * * library.meam Ni Ni.meam Ni
```
The LAMMPS wheel used here does **not** include ML-PACE (ACE, `.yace`) or KIM. Of the machine-learned styles, SNAP and ML-IAP are available.

In [ ]:
%%writefile potential.mod
# Ni, EAM: Y. Mishin, Acta Mater. 52, 1451 (2004)
pair_style   eam/alloy
pair_coeff   * * ni_potentials/NiAl_Mishin_2004.eam.alloy.in Ni
mass         * 58.6934

In [ ]:
POTENTIAL_MOD  = Path('potential.mod')      # passed to every run_lammps() call
POTENTIAL_NAME = 'Mishin_2004'              # label for the results file
results = {'potential': POTENTIAL_NAME}
print(POTENTIAL_MOD.read_text())

**Optional: use your own potential file (Colab).** Run the cell below and pick the file(s) in the dialog. Uploaded files are saved in the current folder,
so in `potential.mod` refer to them by file name, e.g. `pair_coeff * * my_potential.eam.alloy Ni`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('uploaded:', list(uploaded))

### 0.5 Helper: running a LAMMPS script from Python
`run_lammps(script, potential_mod, variables)` starts a serial LAMMPS instance and passes the potential file and the script inputs as `-var` definitions.
It then executes the script with `L.file()`. Numbers are read back with `L.extract_variable()`. Structures come back as files: scripts that relax a structure end with
`write_data ${out_file}`, and the notebook reads that file with ASE (0.6).

**Conventions shared by all scripts**
* Each script uses `include ${potential_file}` after the box is created, so the scripts themselves are independent of the potential.
* Scripts declare their inputs as `variable <name> index <default>`. A value passed with `-var` takes precedence, so every script can also be run on its own:
  `!lmp -in scripts/a0.in -var potential_file potential.mod -var data_file work/a0.data`.
* Units are `metal` throughout.

In [ ]:
def run_lammps(script, potential_mod, variables=None, log=None):
    script, potential_mod = SCRIPT_DIR / script, Path(potential_mod)
    for f in (script, potential_mod):
        if not f.exists():
            raise FileNotFoundError(f)
    args = ['-screen', 'none', '-log', str(log or WORK / f'{script.stem}.log'),
            '-var', 'potential_file', str(potential_mod)]
    for k, v in (variables or {}).items():
        args += ['-var', k, str(v)]
    L = LAMMPS(cmdargs=args)
    L.file(str(script))
    return L          # caller extracts results, then L.close()

### 0.6 Structures: ASE in and out of LAMMPS, OVITO for analysis and pictures
Every structure in the notebook is an ASE `Atoms` object. The helpers below are thin wrappers around library calls:

| helper | what it calls |
|---|---|
| `to_lammps(atoms, path)` | `ase.io.write(..., format='lammps-data')`: the file a script reads with `read_data ${data_file}` |
| `from_lammps(path, pbc)` | `ase.io.read(..., format='lammps-data')` on the file a script wrote with `write_data ${out_file}`. A data file does not store the boundary conditions, so they are set here. |
| `am_to_ase(system)` | atomman → ASE, for the defect structures that atomman builds (stacking-fault slab, dislocation) |
| `pipeline(atoms, *modifiers)` | an OVITO pipeline on the structure; `.compute()` returns the analysed data |
| `render(structures, titles, modifiers, view)` | draws one or more structures side by side as images (OVITO Tachyon renderer, which needs no display) |
| `view3d(atoms, *modifiers)` | an interactive 3D view (OVITO Jupyter widget): drag to rotate, scroll to zoom, hover over an atom to see its properties |

Two OVITO modifiers do the analysis:
* `CNA()`: adaptive common neighbour analysis. Each atom gets a `Structure Type`: 0 other, 1 fcc, 2 hcp, 3 bcc, 4 icosahedral.
  It picks the neighbour cutoff per atom, so no cutoff has to be chosen. Default colours: fcc green, hcp red, bcc blue, other white.
* `DXA(...)`: the dislocation extraction algorithm. It finds dislocation lines and their Burgers vectors (Section 7).

**Frame note.** atomman centres its dislocation cell on the origin, while ASE (and the LAMMPS data file) start the cell at 0.
`am_to_ase()` therefore shifts all positions by $-$`system.box.origin`. Anything computed in atomman's frame, such as the fixed-region bounds
or the slip-plane position, must be shifted by the same amount (Sections 7 and 8).

In [ ]:
def to_lammps(atoms, path):
    ase.io.write(path, atoms, format='lammps-data', atom_style='atomic', masses=True, specorder=['Ni'])
    return path

def from_lammps(path, pbc):
    atoms = ase.io.read(path, format='lammps-data', atom_style='atomic')     # atoms sorted by LAMMPS id
    atoms.pbc = pbc
    return atoms

def am_to_ase(system):
    return system.dump('ase_Atoms')            # positions shifted by -system.box.origin

def pipeline(atoms, *modifiers, show_cell=True):
    data = ase_to_ovito(atoms)
    data.cell.vis.enabled = show_cell     # without the cell outline, zoom_all() fits the atoms
    p = Pipeline(source=StaticSource(data=data))
    for m in modifiers:
        p.modifiers.append(m)
    return p

def structure_counts(data):
    t = np.asarray(data.particles['Structure Type'])
    return {CNA.Type(k).name.lower(): int((t == k).sum()) for k in np.unique(t)}

def render(structures, titles=None, modifiers=(), view='Ortho', size=(640, 480), show_cell=True, camera_dir=None):
    # view: an OVITO viewport type - 'Top' looks down z, 'Front' along y, 'Left' along x, 'Ortho' is oblique
    # show_cell=False: no cell outline, and the camera frames only the atoms left after the modifiers
    # camera_dir: optional viewing direction (x, y, z), overrides the one given by `view`
    structures = list(structures) if isinstance(structures, (list, tuple)) else [structures]
    fig, axes = plt.subplots(1, len(structures), figsize=(5.5 * len(structures), 4.5), squeeze=False)
    for k, (atoms, ax) in enumerate(zip(structures, axes[0])):
        for p in list(ovito.scene.pipelines):          # one structure in the scene at a time
            p.remove_from_scene()
        p = pipeline(atoms, *modifiers, show_cell=show_cell)
        p.add_to_scene()
        vp = Viewport(type=getattr(Viewport.Type, view))
        if camera_dir is not None:
            vp.camera_dir = camera_dir
        vp.zoom_all()
        if not show_cell:                              # close-up: frame the atoms that are left after the modifiers
            pos = np.asarray(p.compute().particles.positions)
            vp.camera_pos = (pos.min(0) + pos.max(0)) / 2
            vp.fov = 0.6 * np.ptp(pos, axis=0).max()   # half-size of the visible area, Å (orthographic views)
        png = WORK / f'render_{k}.png'
        vp.render_image(filename=str(png), size=size, renderer=TachyonRenderer())
        p.remove_from_scene()
        ax.imshow(plt.imread(png)); ax.axis('off')
        title = titles[k] if titles else ''
        data = p.compute()
        if 'Structure Type' in data.particles:
            title += '\n' + ', '.join(f'{n}: {c}' for n, c in structure_counts(data).items())
        ax.set_title(title, fontsize=9)
    plt.show()

def view3d(atoms, *modifiers, height='450px'):
    scene = ovito.Scene()                      # a scene of its own, so later render() calls do not clear this view
    scene.pipelines.append(pipeline(atoms, *modifiers))
    vp = Viewport(scene=scene.scene_root, type=Viewport.Type.Perspective)
    vp.zoom_all()
    return ovito.gui.create_ipywidget(vp, picking=True, layout=ipywidgets.Layout(width='100%', height=height))

try:                                    # Colab shows third-party widgets only after this call
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

## 1. Lattice parameter $a_0$
ASE builds an `NCELL`³ fcc supercell at a guessed lattice parameter (`ase.build.bulk`). It is written to a LAMMPS data file, read with `read_data`,
and relaxed at zero pressure (`fix box/relax iso 0.0`). The script writes the relaxed cell with `write_data ${out_file}`, which is read back and shown next to the initial one.

**Script contract - `a0.in`**

| in (`-var`) | out (`extract_variable`) |
|---|---|
| `data_file`, `ncell`, `out_file` | `a0` [Å] (= `lx/ncell`), `Ecoh` [eV/atom]; relaxed cell in `out_file` |

In [ ]:
A_GUESS, NCELL = 3.52, 4

bulk_initial = ase.build.bulk('Ni', 'fcc', a=A_GUESS, cubic=True).repeat(NCELL)   # NCELL^3 conventional cells
print(bulk_initial)
render(bulk_initial, titles=[f'initial fcc cell, a = {A_GUESS} Å'])

In [ ]:
L = run_lammps('a0.in', POTENTIAL_MOD, {'data_file': to_lammps(bulk_initial, WORK / 'a0.data'),
                                       'ncell': NCELL, 'out_file': WORK / 'a0_out.data'})
a0, Ecoh = L.extract_variable('a0'), L.extract_variable('Ecoh')
p_res = L.get_thermo('press')
L.close()
bulk_relaxed = from_lammps(WORK / 'a0_out.data', pbc=True)

results.update(a0=a0, Ecoh=Ecoh)
print(f'a0 = {a0:.4f} Å   Ecoh = {Ecoh:.4f} eV/atom   residual pressure = {p_res:.2e} bar')
render([bulk_initial, bulk_relaxed], titles=[f'initial, a = {A_GUESS} Å', f'relaxed, a0 = {a0:.4f} Å'], modifiers=[CNA()])

ucell_ase = ase.build.bulk('Ni', 'fcc', a=a0, cubic=True)   # conventional fcc cell at the relaxed a0
ucell = am.load('ase_Atoms', ucell_ase)                     # atomman copy for the defect builders (Sections 5, 7, 8)

**Interactive view of the relaxed perfect cell**, coloured by CNA (every atom is fcc, green). Drag to rotate, scroll to zoom, hover over an atom to see its properties.

In [ ]:
view3d(bulk_relaxed, CNA())

## 2. Burgers vectors from $a_0$
Perfect dislocation: $\mathbf b = \tfrac{a_0}{2}[1\bar10]$, $|\mathbf b| = a_0/\sqrt2$.
Shockley partials: $\mathbf b = \mathbf b_{p1} + \mathbf b_{p2} = \tfrac{a_0}{6}[2\bar1\bar1] + \tfrac{a_0}{6}[1\bar21]$, $|\mathbf b_p| = a_0/\sqrt6$.
(111) interplanar spacing: $d_{111} = a_0/\sqrt3$.

In [ ]:
b_uvw   = np.array([1, -1, 0]) / 2       # crystal (uvw) coordinates, in units of a0
bp1_uvw = np.array([2, -1, -1]) / 6
bp2_uvw = np.array([1, -2, 1]) / 6
assert np.allclose(bp1_uvw + bp2_uvw, b_uvw)

b, bp, d111 = a0 / np.sqrt(2), a0 / np.sqrt(6), a0 / np.sqrt(3)
results.update(b=b, b_p=bp, d111=d111)
print(f'|b| = {b:.4f} Å   |b_p| = {bp:.4f} Å   d111 = {d111:.4f} Å')

## 3. Elastic constants
**Script contract - `elastic.in`**

| in | out |
|---|---|
| `data_file` (fcc supercell at $a_0$) | `C11`, `C12`, `C44` [GPa] |

Cubic symmetry means three independent constants. The values are stored in an `atomman.ElasticConstants` object, which is used in every later step.

In [ ]:
L = run_lammps('elastic.in', POTENTIAL_MOD, {'data_file': to_lammps(ucell_ase.repeat(NCELL), WORK / 'elastic.data')})
Cij_GPa = {k: L.extract_variable(k) for k in ('C11', 'C12', 'C44')}
L.close()

C = am.ElasticConstants(**{k: uc.set_in_units(v, 'GPa') for k, v in Cij_GPa.items()})
results.update(Cij_GPa)
print({k: round(v, 1) for k, v in Cij_GPa.items()}, 'GPa')
print(np.round(uc.get_in_units(C.Cij, 'GPa'), 1))

## 4. Shear modulus from the elastic constants
* Isotropic averages: Voigt, Reuss and Hill estimates, from `C.shear()` and `C.bulk()`. From these, $\nu = \dfrac{3K-2\mu}{2(3K+\mu)}$.
* Shear modulus on the slip system $\{111\}\langle110\rangle$: $\mu_{111} = (C_{11}-C_{12}+C_{44})/3$.
* Zener anisotropy ratio: $A = 2C_{44}/(C_{11}-C_{12})$. Ni is strongly anisotropic, with $A$ of about 2.5, which is why Section 6 also gives the anisotropic estimate.

In [ ]:
mu = {s: uc.get_in_units(C.shear(s), 'GPa') for s in ('Voigt', 'Reuss', 'Hill')}
K  = uc.get_in_units(C.bulk('Hill'), 'GPa')
nu = (3 * K - 2 * mu['Hill']) / (2 * (3 * K + mu['Hill']))
c11, c12, c44 = (Cij_GPa[k] for k in ('C11', 'C12', 'C44'))
mu111, A = (c11 - c12 + c44) / 3, 2 * c44 / (c11 - c12)

results.update(mu_Voigt=mu['Voigt'], mu_Reuss=mu['Reuss'], mu_Hill=mu['Hill'],
               K_Hill=K, nu_Hill=nu, mu_111=mu111, Zener_A=A)
for k in ('mu_Voigt', 'mu_Reuss', 'mu_Hill', 'mu_111', 'K_Hill', 'nu_Hill', 'Zener_A'):
    print(f'{k:9s} = {results[k]:8.3f}')

## 5. Generalized stacking-fault energy
atomman builds a (111) slab and generates the rigidly shifted configurations on a grid of fractional shifts $(a_1, a_2)$
along $\mathbf a_1 = \tfrac12[1\bar10]$ and $\mathbf a_2 = \tfrac12[10\bar1]$. LAMMPS relaxes each configuration perpendicular to the fault only.
The SDVPN model in Section 8 needs this full 2D γ-surface; the 1D curve along $[2\bar1\bar1]$ is a cut through it.

**Script contract - `gsf.in`** (called once per shift)

| in | out |
|---|---|
| `data_file`, `out_file` | `E_total` [eV]; relaxed slab in `out_file` |

The slab is periodic in-plane and has free surfaces along z (`boundary p p s`). Atoms are relaxed along z only (`fix setforce 0 0 NULL`).

In [ ]:
NUM_A1 = NUM_A2 = 12       # γ-surface grid (12x12 = 144 small relaxations); multiple of 6 so 1/6, 1/3, 2/3 are on-grid
SLAB_MINWIDTH = 40.0       # Å, slab thickness normal to (111)

a1_uvw, a2_uvw = np.array([1, -1, 0]) / 2, np.array([1, 0, -1]) / 2
sf = am.defect.StackingFault('1 1 1', ucell, a1vect_uvw=a1_uvw, a2vect_uvw=a2_uvw)
sf.surface(minwidth=SLAB_MINWIDTH, even=True)
print('slab atoms:', sf.system.natoms, '| fault area: %.3f Å^2' % sf.surfacearea)

In [ ]:
a1s, a2s, Es = [], [], []
for a1, a2, system in sf.iterfaultmap(num_a1=NUM_A1, num_a2=NUM_A2):
    L = run_lammps('gsf.in', POTENTIAL_MOD, {'data_file': to_lammps(am_to_ase(system), WORK / 'gsf.data'),
                                            'out_file': WORK / 'gsf_out.data'}, log='none')
    a1s.append(a1); a2s.append(a2); Es.append(L.extract_variable('E_total'))
    L.close()

E0 = Es[0]                               # (a1, a2) = (0, 0): perfect stacking
assert a1s[0] == 0 and a2s[0] == 0
E_gsf = (np.array(Es) - E0) / sf.surfacearea      # eV/Å^2
gamma = am.defect.GammaSurface(a1vect=a1_uvw, a2vect=a2_uvw, box=ucell.box,
                               a1=a1s, a2=a2s, E_gsf=E_gsf)

In [ ]:
fig = gamma.E_gsf_surface_plot(energyperarea_unit='mJ/m^2')
fig = gamma.E_gsf_line_plot(vect=3 * bp1_uvw, energyperarea_unit='mJ/m^2')   # 0 -> isf -> ... along [2-1-1]

In [ ]:
# b_p1 = (a1 + a2)/3. Whether +b_p1 or -b_p1 leads to the intrinsic fault depends on the stacking
# sense of the slab, so evaluate both and keep the side that ends in the lower-energy minimum (the ISF).
s = np.linspace(0, 1, 201)                                   # fraction of |b_p|
paths = {sign: uc.get_in_units(gamma.E_gsf(a1=(sign * s / 3) % 1, a2=(sign * s / 3) % 1), 'mJ/m^2')
         for sign in (+1, -1)}
sign = min(paths, key=lambda k: paths[k][-1])
g_path = paths[sign]
gamma_usf, s_usf = g_path.max(), s[g_path.argmax()]
gamma_isf = g_path[-1]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(s, g_path); ax.axhline(gamma_isf, ls=':', c='k')
ax.set(xlabel=r'shift / $|b_p|$ along ' + ('+' if sign > 0 else '-') + r'$b_{p1}$', ylabel=r'$\gamma$ (mJ/m$^2$)')
results.update(gamma_usf=gamma_usf, gamma_isf=gamma_isf)
print(f'γ_usf = {gamma_usf:.1f} mJ/m² at {s_usf:.2f} b_p  |  γ_isf = {gamma_isf:.1f} mJ/m²')

**Looking at the intrinsic stacking fault.** The slab is shown perfect and with the shift $\mp\mathbf b_{p1}$, both after relaxation, coloured by CNA.
The intrinsic stacking fault changes the stacking from …ABCABC… to …ABC|BCA…: one ABC sequence loses a layer. This leaves two (111) layers with hcp coordination (red).
Atoms at the free surfaces of the slab are classed as *other* (white). The view looks along y, with the slab normal z pointing up. For display only, the slab is repeated 4×4 in-plane.

In [ ]:
a_isf = (sign / 3) % 1
slabs = {}
for name, (f1, f2) in {'perfect': (0.0, 0.0), 'ISF': (a_isf, a_isf)}.items():
    L = run_lammps('gsf.in', POTENTIAL_MOD, {'data_file': to_lammps(am_to_ase(sf.fault(a1=f1, a2=f2)), WORK / 'gsf.data'),
                                            'out_file': WORK / f'gsf_{name}.data'}, log='none')
    L.close()
    slabs[name] = from_lammps(WORK / f'gsf_{name}.data', pbc=(True, True, False)).repeat((4, 4, 1))
render(list(slabs.values()), titles=list(slabs), modifiers=[CNA()], view='Front')

**Interactive view of the intrinsic stacking fault.** The relaxed ISF slab, repeated 4×4 in-plane and coloured by CNA: the two red (hcp) (111) layers are the fault, the white atoms are the free surfaces.
Rotate it to look along the fault plane and count the stacking sequence.

In [ ]:
view3d(slabs['ISF'], CNA())

## 6. Equilibrium partial separation
The two Shockley partials repel elastically and are pulled together by the **intrinsic** stacking fault between them. They settle where the two forces balance.
($\gamma_{usf}$ is the barrier for *nucleating* a partial. It enters the dislocation core and Peierls stress through the SDVPN model in Section 8, not $d_{eq}$.)

The partials repel with a force per unit length $\mu\,(\mathbf b_{p1}\cdot\mathbf b_{p2})/(2\pi d)$, and $\mathbf b_{p1}\cdot\mathbf b_{p2} = b^2/6$.
Setting this equal to the stacking-fault energy $\gamma_{sf}$ (here $\gamma_{sf} = \gamma_{isf}$ from Section 5) gives
$$d_{sf} = \frac{\mu b^2}{12\pi\gamma_{sf}}$$


In [ ]:
def d_sf(mu_GPa, b, g_mJ):
    return uc.set_in_units(mu_GPa, 'GPa') * b**2 / (12 * np.pi * uc.set_in_units(g_mJ, 'mJ/m^2'))

results['d_sf_iso'] = d_sf(mu['Hill'], b, gamma_isf)
print(f"d_sf = {results['d_sf_iso']:6.1f} Å   (all characters)")

## 7. Periodic array of dislocations (PAD)
> **Not validated yet:** the code cells of Sections 7 and 8 are commented out.

atomman inserts one edge dislocation ($\mathbf b=\tfrac{a_0}{2}[1\bar10]$, $\boldsymbol\xi=[11\bar2]$, slip plane (111)) using the anisotropic Volterra solution.
The cell is periodic along $x$ ($\mathbf m$, glide direction) and $z$ ($\boldsymbol\xi$), and has free surfaces along $y$ ($\mathbf n$, slip-plane normal).
atomman marks the atoms within `PAD_BOUNDARY` of the $y$ surfaces as atom type 2. In the ASE structure every atom is Ni, so the data file has a single atom type
and the same `potential.mod` works. The script holds the boundary atoms fixed by position instead: everything outside `ylo_free` < y < `yhi_free`.
`base` is the matching perfect crystal, which Section 8 needs as the reference.

**Script contract - `pad.in`**

| in | out |
|---|---|
| `data_file`, `ylo_free`, `yhi_free` [Å], `out_file` | `E_total` [eV]; relaxed cell in `out_file` |

The cell is `boundary p s p`, and atoms outside the free slab are held fixed during minimization (optionally followed by a short MD anneal).

In [ ]:
# PAD_AMIN, PAD_BMIN, PAD_CMIN = 200.0, 100.0, 10.0     # Å along x (m), y (n), z (ξ)
# PAD_BOUNDARY = 10.0                                  # Å of fixed atoms at each y surface
#
# dislocation = am.defect.Dislocation(ucell, C, burgers=b_uvw, ξ_uvw=[1, 1, -2], slip_hkl=[1, 1, 1],
#                                     m=[1, 0, 0], n=[0, 1, 0])
# base, pad = dislocation.periodicarray(amin=PAD_AMIN, bmin=PAD_BMIN, cmin=PAD_CMIN,
#                                       boundarywidth=PAD_BOUNDARY, return_base_system=True)
# assert np.allclose(base.box.origin, pad.box.origin)
# fixed    = pad.atoms.atype == 2                      # boundary atoms (same order in every structure below)
# pad_ase  = am_to_ase(pad)                            # frame shifted by -pad.box.origin, see 0.6
# base_ase = am_to_ase(base)
# print('PAD atoms:', len(pad_ase), '| fixed boundary atoms:', int(fixed.sum()), '| cell (Å):', pad_ase.cell.lengths().round(2))
# render(pad_ase, titles=['PAD as built (Volterra solution)'], modifiers=[CNA()], view='Top')

In [ ]:
# free_y = pad_ase.positions[~fixed, 1]                # already in the ASE/LAMMPS frame
# L = run_lammps('pad.in', POTENTIAL_MOD, {'data_file': to_lammps(pad_ase, WORK / 'pad.data'), 'out_file': WORK / 'pad_out.data',
#                                         'ylo_free': free_y.min() - 0.01, 'yhi_free': free_y.max() + 0.01})
# results['E_pad'] = L.extract_variable('E_total')
# L.close()
# pad_relaxed = from_lammps(WORK / 'pad_out.data', pbc=pad_ase.pbc)
# assert len(pad_relaxed) == len(base_ase)

**Before and after relaxation, defect atoms only.** Atoms classed as fcc by CNA, and all atoms more than 15 Å from the slip plane, are hidden.
As built, the Volterra solution is a single compact core. After relaxation it splits into two Shockley partials (cores in white and blue: other and bcc) joined by a ribbon of hcp atoms (red),
which is the intrinsic stacking fault from Section 5. The view looks down the line direction $z$, zoomed in on the defect atoms.

**First estimate of $d_{eq}$: the width of the hcp ribbon.** It is direct and intuitive, but rough: CNA assigns each atom to one class,
so the ribbon edges are only resolved to about one atomic spacing, and core atoms (white) are not counted.

In [ ]:
# data = {'as built': pipeline(pad_ase, CNA()).compute(), 'relaxed': pipeline(pad_relaxed, CNA()).compute()}
# y_slip = -pad.box.origin[1]                          # slip plane (y = 0 in atomman's frame), see 0.6
# hide = [ExpressionSelectionModifier(expression=f'StructureType == 1 || abs(Position.Y - {y_slip}) > 15'),
#         DeleteSelectedModifier()]                    # keep only non-fcc atoms within 15 Å of the slip plane
# render([pad_ase, pad_relaxed], titles=list(data), modifiers=[CNA(), *hide], view='Top', show_cell=False)
#
# st = np.asarray(data['relaxed'].particles['Structure Type'])
# x_hcp = pad_relaxed.positions[(st == CNA.Type.HCP) & ~fixed, 0]
# results['d_eq_PAD_cna'] = np.percentile(x_hcp, 98) - np.percentile(x_hcp, 2) if len(x_hcp) else 0.0
# print(f"hcp ribbon width (2nd-98th percentile in x): {results['d_eq_PAD_cna']:.1f} Å")

**Second estimate: dislocation extraction (DXA).** DXA builds a Burgers circuit around every defect region and returns the dislocation lines with
their Burgers vectors. It should find two Shockley partials, $\mathbf b_p = \tfrac16\langle112\rangle$ (printed in crystal coordinates, units of $a_0$),
and their spacing is $d_{eq}$. This is the measure normally reported in the literature.

In the picture, OVITO draws Shockley partials as green lines. The view looks obliquely onto the slip plane, and the atoms are drawn see-through,
so the two lines are visible along the two edges of the hcp ribbon.

In [ ]:
# dxa = pipeline(pad_relaxed, DXA(input_crystal_structure=DXA.Lattice.FCC)).compute()
# segments = list(dxa.dislocations.segments)
# for s in segments:
#     print(f'b = {np.round(s.true_burgers_vector, 3)}   length {s.length:.1f} Å   mean x {s.points[:, 0].mean():.2f} Å')
# x_lines = sorted(s.points[:, 0].mean() for s in segments)
# if len(segments) != 2:
#     print(f'WARNING: DXA found {len(segments)} segments, expected 2 Shockley partials')
# results['d_eq_PAD_dxa'] = x_lines[-1] - x_lines[0] if len(segments) >= 2 else 0.0
# print(f"DXA partial separation: {results['d_eq_PAD_dxa']:.1f} Å   (hcp ribbon: {results['d_eq_PAD_cna']:.1f} Å)")
# def dxa_for_drawing():
#     m = DXA(input_crystal_structure=DXA.Lattice.FCC)
#     m.defect_vis.enabled = False             # do not draw the defect surfaces (here: the two free surfaces)
#     m.disloc_vis.line_width = 1.5              # Å
#     return m
#
# render(pad_relaxed, titles=['relaxed PAD: DXA lines (Shockley partials) and the hcp ribbon'],
#        modifiers=[dxa_for_drawing(), *hide,
#                   ComputePropertyModifier(output_property='Transparency', expressions=['0.75'])],  # see-through atoms
#        view='Ortho', show_cell=False, camera_dir=(0.35, -1, -0.45))     # obliquely onto the slip plane

**Optional: interactive 3D view.** Drag to rotate, scroll to zoom. On Google Colab, uncomment the first two lines.
The view disappears when another `render()` call runs.

In [ ]:
# # from google.colab import output
# # output.enable_custom_widget_manager()
# for p in list(ovito.scene.pipelines):
#     p.remove_from_scene()
# pipeline(pad_relaxed, dxa_for_drawing(), *hide, show_cell=False).add_to_scene()
# vp = Viewport(type=Viewport.Type.Ortho)
# vp.zoom_all()
# vp.create_jupyter_widget()

## 8. Disregistry and CRSS
### 8.1 Atomistic disregistry of the relaxed PAD
$\delta(x) = \mathbf u_{above}(x) - \mathbf u_{below}(x)$ across the slip plane. The two partials show up as two peaks in the dislocation density $\rho = d\delta/dx$, and their spacing is the simulated $d_{eq}$.

In [ ]:
# # atomman needs the reference and the relaxed structure in the same frame and atom order: convert both ASE structures back.
# base_am, pad_relaxed_am = am.load('ase_Atoms', base_ase), am.load('ase_Atoms', pad_relaxed)
# # slip plane: midway between the two (111) planes closest to the dislocation. That is y = 0 in atomman's PAD frame,
# # which am_to_ase() shifted by -pad.box.origin (see 0.6).
# planepos = -pad.box.origin
# x_md, dis_md = am.defect.disregistry(base_am, pad_relaxed_am, m=[1, 0, 0], n=[0, 1, 0], planepos=planepos)
#
# def partial_positions(x, dis):
#     rho = np.gradient(dis[:, 0], x)                  # edge component
#     order = np.argsort(rho)[::-1]
#     p1 = x[order[0]]
#     p2 = next(x[i] for i in order[1:] if abs(x[i] - p1) > 2 * bp)   # second peak, not adjacent
#     return sorted((p1, p2))
#
# p_md = partial_positions(x_md, dis_md)
# results['d_eq_PAD'] = p_md[1] - p_md[0]
#
# fig, ax = plt.subplots(figsize=(7, 3.5))
# ax.plot(x_md, dis_md[:, 0] / b, 'o', ms=3, label=r'$\delta_x$ (edge)')
# ax.plot(x_md, dis_md[:, 2] / b, 's', ms=3, label=r'$\delta_z$ (screw)')
# for p in p_md: ax.axvline(p, ls=':', c='k')
# ax.set(xlabel='x (Å)', ylabel=r'$\delta / |b|$', title=f"PAD: d_eq = {results['d_eq_PAD']:.1f} Å"); ax.legend()

### 8.2 Semidiscrete variational Peierls-Nabarro (SDVPN)
The SDVPN model minimizes the elastic energy plus the misfit energy from the γ-surface (Section 5) on a **discrete** grid. The grid spacing is the
spacing of atomic columns along $\mathbf m$, which is $|\mathbf b|/2$ for this edge dislocation. The discreteness is what produces a finite Peierls stress.

In [ ]:
# XSTEP = b / 2
# XNUM  = 2 * int(100.0 / XSTEP)                       # grid points (x spans about ±100 Å)
#
# volterra = am.defect.solve_volterra_dislocation(C, b_uvw * a0, ξ_uvw=[1, 1, -2], slip_hkl=[1, 1, 1],
#                                                 box=ucell.box, m=[1, 0, 0], n=[0, 1, 0])
# x, dis0 = am.defect.pn_arctan_disregistry(xnum=XNUM, xstep=XSTEP, burgers=volterra.burgers, halfwidth=3 * bp)
#
# pn = am.defect.SDVPN(volterra=volterra, gamma=gamma, cutofflongrange=XNUM * XSTEP / 2,
#                      min_method='L-BFGS-B', min_options={'maxiter': 2000})
# pn.solve(x=x, disregistry=dis0)
#
# p_pn = partial_positions(pn.x, pn.disregistry)
# results['d_eq_SDVPN'] = p_pn[1] - p_pn[0]
# fig = pn.disregistry_plot()
# print(f"SDVPN d_eq = {results['d_eq_SDVPN']:.1f} Å   (PAD: {results['d_eq_PAD']:.1f} Å)")

### 8.3 CRSS (Peierls stress) from SDVPN
Apply an increasing resolved shear stress $\tau_{xy}$ (glide direction $\mathbf m$, plane normal $\mathbf n$), re-solve starting from the previous solution each time, and follow the dislocation centre $\bar x = \sum x\rho / \sum\rho$.
The CRSS is the lowest stress at which the centre no longer settles but moves through the grid.

In [ ]:
# TAUS_MPa = np.arange(0, 205, 5)
# MOVE_TOL = 4 * XSTEP                                 # Å; a displacement this large counts as glide
#
# def centre(x, dis):
#     rho = np.abs(np.gradient(dis[:, 0], x))
#     return (x * rho).sum() / rho.sum()
#
# x_ref = centre(pn.x, pn.disregistry)
# centres, crss = [], None
# dis = pn.disregistry.copy()
# for t in TAUS_MPa:
#     tau = np.zeros((3, 3)); tau[0, 1] = tau[1, 0] = uc.set_in_units(t, 'MPa')
#     pn.solve(x=x, disregistry=dis, tau=tau)
#     dis = pn.disregistry.copy()
#     centres.append(centre(x, dis))
#     if abs(centres[-1] - x_ref) > MOVE_TOL:
#         crss = t; break
#
# results['CRSS_SDVPN_MPa'] = crss
# fig, ax = plt.subplots(figsize=(5, 3.5))
# ax.plot(TAUS_MPa[:len(centres)], np.array(centres) - x_ref, 'o-')
# ax.set(xlabel=r'$\tau_{xy}$ (MPa)', ylabel=r'$\Delta\bar{x}$ (Å)', title=f'CRSS ≈ {crss} MPa')

## Summary

In [ ]:
out = Path(f'results_{POTENTIAL_NAME}.json')
out.write_text(json.dumps({k: (float(v) if isinstance(v, (np.floating, float, int)) and v is not None else v)
                           for k, v in results.items()}, indent=2))
for k, v in results.items():
    print(f'{k:18s} {v:.4g}' if isinstance(v, (float, np.floating)) else f'{k:18s} {v}')